In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

#библиотеки визуализации
import seaborn as sns
import matplotlib.pyplot as plt

import math
from datetime import datetime
from itertools import combinations
from scipy import stats

#библиотеки ML. 
# модели
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier

#преобразования, метрики
from category_encoders import TargetEncoder
from sklearn.preprocessing import StandardScaler, OneHotEncoder,RobustScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split,GridSearchCV,StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve,precision_score,recall_score,f1_score
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


#оптимизация гиперпараметров
from skopt import BayesSearchCV
from skopt.space import Real,Categorical,Integer
import optuna

import warnings
warnings.filterwarnings('ignore')

1. Задача, метрики, данные
   
**Задача** - предсказать заключит ли клиент договор на вклад или нет. То есть бинарная классификация.

**Метрика** Основная метрика качества модели - ROC-AUC. Дополнительная PR-AUC для работы с дисбалансом классов. Поскольку финансовые параметры нам неизвестны, бизнес метрики не применяются.

**Данные** - синтетические основанные на оригинальном банковском датасете. Для чистоты, оригинальные данные в обучении использовать не будем.

2. EDA: распределения, выбросы, корреляции

In [ ]:
#Загружаем датасеты. Тренировочный и тестовый.
df_train = pd.read_csv("/kaggle/input/playground-series-s5e8/train.csv")
df_test  = pd.read_csv("/kaggle/input/playground-series-s5e8/test.csv")

In [ ]:
#посмотрим на случайные 5 наблюдений
df_train.sample(5)

In [ ]:
#Посмотрим общую информацию по датасету.
df_train.info()

Видим 17 признаков и 1 метку таргета. Добавим описание признаков для удобства.

**Описание**

**id**: Идентификационный номер 

**age**: Возраст клиента (числовое значение)

**job**: Работа (категориальный)

**marital**: Семейное положение (категориальный)

**education**: Уровень образования (категориальный)

**default**: Есть ли просроченный кредит? (категориальный)

**balance**: Среднегодовой баланс в евро (числовое значение)

**housing**: Есть ли жилищный кредит? (категориальный)

**loan**: Есть ли личный кредит? (категориальный)

**contact**: Тип связи с клиентом (категориальный)

**day**: День последнего контакта с клиентом (числовое значение)

**month**: Месяц последнего контакта с клиентом  (категориальный)

**duration**: Продолжительность последнего контакта в секундах (числовое значение)

**campaign**: Количество контактов с клиентом в рамках текущей кампании (числовое значение)

**pdays**: Количество дней, прошедших с последнего контакта в предыдущей кампании (числовое значение)

**previous**: Количество контактов до текущей кампании (числовое значение)

**poutcome**: Результат предыдущей маркетинговой кампании (категориальный)

**y**: Целевая переменная, подписался ли клиент на срочный депозит (бинарный)

In [ ]:
#посмотрим на статистику числовых данных
df_train.describe()

**Вывод**

1. Все признаки имеют полные данные, так как count равен 750000 для каждого столбца.
2. Баланс (balance) в среднем около 1204 евро, но большие колебания: максимум до 99717, а минимум до -8019 (возможно, отрицательный баланс).
3. Некоторые признаки, как pdays и previous, имеют отрицательные значения (-1) — это может означать отсутствие предыдущих контактов или другой специальный код.
4. Целевой признак "y" (договор на депозит) имеет среднее 0.12065, что говорит о том, что примерно 12% клиентов подписались.

In [ ]:
#выведем уникальные значения категориальных переменных
for col in df_train.select_dtypes(exclude = ['number']).columns:
    unique_values = df_train[col].unique()
    print(f'Уникальные значения для признака {col} : {unique_values}')

In [ ]:
#есть ли пропуски
df_train.isna().sum()

In [ ]:
#Пропусков в формате NaN нет, но в перечне значений категориальных признаков было "Uncknown" 
for col in df_train.columns:
    if df_train[col].dtype == 'object':
        print(f'Доля неизвестных в {col} : {round(df_train[col].isin(["unknown"]).sum()/df_train.shape[0]*100,2)}%')

Видим что в признаках contact и poutcome много неизвестных. 

Признак poutcome имеет сильно несбалансированное распределение (до 90% unknown), что сильно снижает его информативность и усложняет обучение модели.
    Потому в текущей работе он был удален.

In [ ]:
df_train.drop('poutcome' , axis = 1, inplace = True)
df_test.drop('poutcome' , axis = 1, inplace = True)

In [ ]:
#проверим на дубликаты
num_duplicates = df_train.duplicated().sum()
print(f'Количество дубликатов: {num_duplicates}')

Построим графики распределений

In [ ]:
# Построение графиков распределения 
def plot_feature_target_distributions(df):
    num_features = df.shape[1]
    cols = 2 # Количество столбцов
    rows = math.ceil(num_features / cols)  # Количество строк
    plt.figure(figsize=(cols * 6, rows * 5))
    for i, column in enumerate(df.columns.drop('y')):
        plt.subplot(rows, cols, i + 1)
        sns.histplot(data=df, x=column, hue='y',  stat="probability", common_norm=False, bins=30)
        plt.xticks(rotation=45)
        plt.title(f'Распределение признака: {column}')
    plt.tight_layout()
    plt.show()


plot_feature_target_distributions(df_train)

**Выводы о распределениях**

* числовые признаки распределены логнормально с длинным хвостом.
* при возрасте от 30 до 60 клиент вероятнее не заключит договор.
* blue-collar и services менее склонны делать вклады.
* students,management,retired,unemployed более склонны делать вклады.
* одинокие более часто делают вклады чем семейные.
* люди с высшим образованием намного больше делают вклады.
* с ростом баланса растет вероятность вклада.
* если есть жилищный кредит или займ человек скорее всего не сделает вклад.
* возможно неуказанный способ связи говорит о нежелании клиента сотрудничать, поэтому они с большей вероятностью отказываются.
* часть месяцев сильно выделяются повышенным уровнем согласия или отказов. 
* при очень маленьком duration очень маловероятно заключение договра. Что логично, если разговор не состоялся, видимо клиент не заинтересован.
* если предыдущая компания закончилась успехом, то новая скорее так же будет успешной.

In [ ]:
#Визуализация целевой переменной
sns.countplot(x=target, data=df_train)
plt.title("Distribution of target 'y'")
plt.show()

**Выбросы**

In [ ]:
# Построение boxplot для каждого признака датафрейма
def plot_boxplots(df):
    cols = 3
    rows = (len(df.columns) + cols - 1) // cols
    plt.figure(figsize=(cols * 6, rows * 5))
    for i, col in enumerate(df.columns.drop('y')):
        plt.subplot(rows, cols, i + 1)
        sns.boxplot(x=df[col])
        plt.title(f'Boxplot: {col}')
    plt.tight_layout()
    plt.show()


plot_boxplots(df_train.select_dtypes(include = 'number'))

In [ ]:
#удаление выбросов
def handle_outliers_log(df):
    numeric_cols = df.select_dtypes(include=['number']).columns
    df_clean = df.copy()

    for col in numeric_cols:
        # Преобразование логарифмированием (с отрицательными значениями не трогаем)
        if (df_clean[col] <= 0).any():
            continue

        log_data = np.log(df_clean[col])

        Q1 = log_data.quantile(0.25)
        Q3 = log_data.quantile(0.75)
        IQR = Q3 - Q1

        lower_bound_log = Q1 - 1.5 * IQR
        upper_bound_log = Q3 + 1.5 * IQR

        lower_bound = np.exp(lower_bound_log)
        upper_bound = np.exp(upper_bound_log)

        df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]

    return df_clean

In [ ]:
#проверим количество оставшихся данных
shape_before = df_train.shape[0]
df_train = handle_outliers_log(df_train)
shape_after = df_train.shape[0]
print (f'Осталось {shape_after/shape_before*100}% наблюдений')

In [ ]:
#еще раз взглянем на графики после удаления выбросов
plot_boxplots(df_train.select_dtypes(include = 'number'))

In [ ]:
# Выделение только числовых столбцов
numeric_df = df_train.select_dtypes(include=['number'])
corr_matrix = numeric_df.corr()  # корреляция Пирсона по умолчанию
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", square=True, cbar=True)
plt.title("Матрица корреляций числовых признаков")
plt.show()

* duration и таргет (0.52) — длительность положительно связана с таргетом, возможно, длительность оказывает существенное влияние на результат.

* pdays и previous (0.57) — высокая корреляция указывает, что эти признаки тесно связаны. Это может привести к мультиколлинеарности и сложностям в интерпретации

* Таргет имеет положительные, но не очень сильные корреляции, с balance (0.12), pdays (0.09), previous (0.13).

In [ ]:
# Для экономии памяти и ускорения вычислений переводим числовые признаки к минимально возможным типам
numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64', "uint16", "uint32", "uint64"]
for col in df_train.columns:
            col_type = df_train[col].dtypes
            
            if col_type in numerics:
                c_min = df_train[col].min()
                c_max = df_train[col].max()

                if "int" in str(col_type):
                    if c_min >= np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                        df_train[col] = df_train[col].astype(np.int8)
                    elif c_min >= np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                        df_train[col] = df_train[col].astype(np.int16)
                    elif c_min >= np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                        df_train[col] = df_train[col].astype(np.int32)
                    elif c_min >= np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                        df_train[col] = df_train[col].astype(np.int64)  
                else:
                    if c_min >= np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                        df_train[col] = df_train[col].astype(np.float16)
                    if c_min >= np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                        df_train[col] = df_train[col].astype(np.float32)
                    else:
                        df_train[col] = df_train[col].astype(np.float64)

In [ ]:
#делим на фичи и таргет. Признак id нам не нужен,т.к. не несет информации.
features = df_train.drop(['y','id'],axis = 1)
target = df_train['y']

In [ ]:
#разделям выборки на тренировочную, валидационную и тестовую. 60:20:20 Учитываем дисбаланс классов
# train   – для обучения модели
# val     – для подбора гиперпараметров / валидации
# test    – как прокси продакшн (никак не трогаем при обучении)
features_train,features_val_test,target_train,target_val_test = train_test_split(features,target,random_state = 42, test_size = 0.4,stratify=target)
features_val,features_test,target_val,target_test = train_test_split(features_val_test,target_val_test,random_state = 42, test_size = 0.5,stratify=target_val_test)

In [ ]:
# у степени образованности есть порядковая зависимость. 
#   'unknown' < 'primary' < 'secondary' < 'tertiary'
# используем OrdinalEncoder, чтобы сохранить эту иерархию для модели
education_categories = [ 'unknown', 'primary','secondary', 'tertiary']
ordinal_encoder = OrdinalEncoder(categories = [education_categories])
features_train['education_ordinal'] = ordinal_encoder.fit_transform(features_train[['education']])
features_val['education_ordinal'] = ordinal_encoder.transform(features_val[['education']])
features_test['education_ordinal'] = ordinal_encoder.transform(features_test[['education']])

In [ ]:
# Сделаны только бизнес‑смысловые фичи;  
# слепой перебор комбинаций не используется,  
# чтобы модель оставалась интерпретируемой и не перегружалась.  
# Если в дальнейшем будет нужно,  
# можно добавить ограниченные осознанные комбинации и отфильтровать их по feature_importance.
3. Feature engineering (без слепого перебора)
#создадим дополнительные признаки вручную
def feature_constuction(features):
    #метка положительности баланса
    features['balance_positive'] = (features['balance'] > 0).astype(int)

    #средняя продолжительность кампании
    features['avg_duration_per_campaign'] = features['duration'] / (features['campaign'] + 1)

    #метка что вообще не контактировали
    features['pdays_negative'] = (features['pdays'] == -1).astype(int)

    #соотношение контактов до и сейчас
    features['prev_to_campaign_ratio'] = features['previous'] / (features['campaign'] + 1)

    #обьединенный класс семейного положения и жилья
    features['marital_housing'] = features['marital'] + '_' + features['housing']
    
    #логарифм баланса
    features['balance_log'] = np.sign(features['balance']) * np.log1p(np.abs(features['balance']))
    
    #логарифм длительности
    features['duration_log'] = np.log1p(features['duration'])

    #логарифм количества контактов
    features['campaign_log'] = np.log1p(features['campaign'])

    return features

In [ ]:
# Функция создания новых числовых признаков (произведение пар)
def add_numeric_combinations(features):
    numeric_cols = features.select_dtypes(include='number').columns
    new_cols = {}
    for c1, c2 in combinations(numeric_cols, 2):
        new_cols[f"{c1}_{c2}"] = features[c1] * features[c2]
    new_features = pd.DataFrame(new_cols)
    return pd.concat([features, new_features], axis=1)

In [ ]:
# Функция создания новых категориальных признаков (конкатенация пар)
def add_categorical_combinations(features):
    categorical_cols = features.select_dtypes(exclude='number').columns
    features[categorical_cols] = features[categorical_cols].astype('category')
    new_features = pd.DataFrame()
    for cols in combinations(categorical_cols, 2):
        new_col_name = '_'.join(cols) + '_enc'
        new_features[new_col_name] = features[list(cols)].astype(str).agg('_'.join, axis=1).astype('category')
    return pd.concat([features, new_features], axis=1)

In [ ]:
# Создание различных вариантов датасетов
def create_datasets(features_train, features_val):
    datasets = {}

    base_train = features_train.copy()
    base_val = features_val.copy()
    
    #Признаки созданные вручную добавляем в все варианты
    base_train = feature_constuction(base_train)
    base_val = feature_constuction(base_val)

    # Вариант 1: базовые признаки
    datasets['base'] = (base_train, base_val)

    # Вариант 2: с новыми числовыми признаками
    #num_train = add_numeric_combinations(base_train.copy())
    #num_val = add_numeric_combinations(base_val.copy())
    #datasets['with_numeric'] = (num_train, num_val)

    # Вариант 3: с новыми категориальными признаками
    #cat_train = add_categorical_combinations(base_train.copy())
    #cat_val = add_categorical_combinations(base_val.copy())
    #datasets['with_categorical'] = (cat_train, cat_val)

    # Вариант 4: с новыми числовыми и категориальными признаками
    #all_train = add_numeric_combinations(base_train.copy())
    #all_train = add_categorical_combinations(all_train)
    #all_val = add_numeric_combinations(base_val.copy())
    #all_val = add_categorical_combinations(all_val)
    #datasets['with_all'] = (all_train, all_val)

    return datasets

In [ ]:
def preprocess_and_pca(features_train, features_val, apply_pca=False):
    
    numeric_cols = features_train.select_dtypes(include='number').columns
    categorical_cols = features_train.select_dtypes(exclude='number').columns

    features_train[categorical_cols] = features_train[categorical_cols].astype('category')
    features_val[categorical_cols] = features_val[categorical_cols].astype('category')

    preprocessor = ColumnTransformer([
        ('num', RobustScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse=False), categorical_cols)
    ])

    features_train_processed = preprocessor.fit_transform(features_train)
    features_val_processed = preprocessor.transform(features_val)

    if apply_pca:
        pca = PCA(n_components=0.95)  # 95% вариации
        features_train_processed = pca.fit_transform(features_train_processed)
        features_val_processed = pca.transform(features_val_processed)

    return features_train_processed, features_val_processed

4. Model comparison: все модели
** Обоснование выбора моделей
*LogisticRegression
Используется как бейзлайн. Хорошо интерпретируема.
*DecisionTreeClassifier
Интерпретируемость, может работать без preprocessing.
*BernoulliNB
Быстрый, но обладает сильным предположением о независимости признаков.
При наличии корреляций и сложных зависимостей он будет хуже.
*SVM не используется
Большое время обучения.
Плохо масштабируется на большие датасеты 
*MLPClassifier (нейронная сеть)
Может работать с нелинейностями, но
не всегда даёт прирост по сравнению с XGB в табличных задачах и
более сложен в интерпретации и поддержке.
*RandomForestClassifier
Плюсы:
Устойчив к выбросам,
хорош на табличных задачах,
хорошо работает с дисбалансом.
*XGBClassifier
Хорош в табличных задачах,
хорошо работает с дисбалансом (scale_pos_weight),
масштабируемость достаточная для продакшн‑размеров.
наибольший потенциал при оптимизации гиперпараметров,
интерпретируемость через feature_importance.
    

In [ ]:
# Инициализация моделей
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(),
    'Bernoulli Naive Bayes': BernoulliNB(),
    'Neural Network (MLP)': MLPClassifier(max_iter=1000),
    'Random Forest': RandomForestClassifier(),
    'XGBClassifier':xgb.XGBClassifier()
}

In [ ]:
results = []
datasets = create_datasets(features_train, features_val)

for data_name, (X_train, X_val) in datasets.items():
    for use_pca in [False, True]:
        print(f"Обучение на данных '{data_name}' с применением PCA = {use_pca}")
        X_train_final, X_val_final = preprocess_and_pca(X_train, X_val, apply_pca=use_pca)
        for model_name, model in models.items():
            print(f"  Обучение модели: {model_name}")
            model.fit(X_train_final, target_train)
            y_pred = model.predict(X_val_final)

            if hasattr(model, "predict_proba"):
                y_proba = model.predict_proba(X_val_final)[:, 1]
                roc_auc = roc_auc_score(target_val, y_proba)
            else:
                roc_auc = None

            acc = accuracy_score(target_val, y_pred)
            prec = precision_score(target_val, y_pred, zero_division=0)
            rec = recall_score(target_val, y_pred, zero_division=0)
            f1 = f1_score(target_val, y_pred, zero_division=0)

            results.append({
                'Dataset': data_name,
                'PCA Applied': use_pca,
                'Model': model_name,
                'Accuracy': acc,
                'Precision': prec,
                'Recall': rec,
                'F1 Score': f1,
                'ROC AUC': roc_auc
            })

In [ ]:
# ищем лучшие результаты по ROC AUC
df_results = pd.DataFrame(results)
df_results.sort_values(by = 'ROC AUC',ascending = False)

# Выводы
Лучшие результаты у градиентного бустинга, независимо от созданных новых признаков, без PCA.

За градиентным бустингом идут случайный лес и нейронная сеть.

Модели на данных без PCA показали себя в целом лучше.

5. Final model: XGB + Optuna

Для дальнейшей работы попробуем оптимизировать гиперпараметры для градиентного бустинга на различных комбинациях данных и сравним их.

In [ ]:
neg_count = sum(target_train == 0)
pos_count = sum(target_train == 1)
scale_pos_weight = neg_count / pos_count

X_train, X_val = datasets['base']
X_train_final, X_val_final = preprocess_and_pca(X_train, X_val, apply_pca=False)

In [ ]:
def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 30),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
        "subsample": trial.suggest_float("subsample", 0.1, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.01, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 2, 20),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 15),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 10.0, log=True),
        "scale_pos_weight": scale_pos_weight
    }

    model = xgb.XGBClassifier(**params)
    model.fit(X_train_final, target_train)

    y_pred_proba = model.predict_proba(X_val_final)[:, 1]
    roc_auc = roc_auc_score(target_val, y_pred_proba)

    if trial.should_prune():
        raise optuna.exceptions.TrialPruned()

    return roc_auc

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Лучшие параметры:", study.best_params)
print("Лучший ROC-AUC:", study.best_value)

optuna.importance.get_param_importances(study)

best_xgb = xgb.XGBClassifier(**study.best_params)

features_train_val = pd.concat([features_train, features_val], ignore_index=True)
target_train_val = pd.concat([target_train, target_val], ignore_index=True)

features_train_final, features_test_final = preprocess_and_pca(features_train_val, features_test, apply_pca=False)

best_xgb.fit(features_train_final,target_train_val )
xgb_y_pred_proba = best_xgb.predict_proba(features_test_final)[:, 1]

6. Metric analysis: ROC‑AUC + PR‑AUC + precision/recall/F1

In [ ]:
# ROC-AUC
xgb_roc_auc = roc_auc_score(target_test, xgb_y_pred_proba)

# В условиях сильного дисбаланса классов дополнительно к ROC‑AUC
# используем PR‑oriented метрики, которые лучше отражают качество
# именно на положительном (редком) классе:

In [ ]:
# 1) PR‑AUC (average precision)
pr_auc = average_precision_score(target_test, xgb_y_pred_proba)

# 2) Precision, Recall, F1 на выбранном пороге
threshold = 0.5  # можно подобрать, например, через PR‑кривую
y_pred = (xgb_y_pred_proba >= threshold).astype(int)

precision = precision_score(target_test, y_pred)
recall = recall_score(target_test, y_pred)
f1 = f1_score(target_test, y_pred)

print(f'ROC‑AUC:        {xgb_roc_auc:.4f}')
print(f'PR‑AUC:         {pr_auc:.4f}')
print(f'Precision@{threshold:.2f}: {precision:.4f}')
print(f'Recall@{threshold:.2f}:    {recall:.4f}')
print(f'F1@{threshold:.2f}:        {f1:.4f}')

### 7. Итоговый выбор модели и следующие шаги

- Лучшую метрику ROC‑AUC показала модель XGBClassifier.
- Random Forest и MLP закрепились на втором месте.
- Лучшие результаты получены на:
  - варианте `'base'` (без дополнительных комбинаций),
  - без PCA.
- Финальная модель:  
  XGBClassifier с подобранными параметрами через Optuna.
- Дальнейшие возможные шаги:
  - подбор оптимального порога по PR‑кривой,  
  - добавление ограниченных, бизнес‑смысловых комбинаций фич